In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import openreview
from collections import defaultdict
import json
import os

import pickle

In [3]:
client_v1 = openreview.Client(
    baseurl='https://api.openreview.net',
    username="",
    password=""
)

client_v2 = openreview.api.OpenReviewClient(
        baseurl='https://api2.openreview.net',
        username="",
        password=""
    )

In [19]:
def crawlAPIv2(year, conference):
    assert conference in ["ICLR", "NeurIPS"], "Only ICLR and NeurIPS are supported."

    venue_id = f"{conference}.cc/{year}/Conference"
    venue_group = client_v2.get_group(venue_id)
    submission_name = venue_group.content['submission_name']['value']

    submissions = client_v2.get_all_notes(invitation=f'{venue_id}/-/{submission_name}', details='replies')

    assert len(submissions) > 0, "No submissions found. Something is wrong..."

    dataDir = "openReviewData/raw"

    with open(f'{dataDir}/{conference}{year}.pkl', 'wb') as f:
        pickle.dump(submissions, f)

def crawlAPIv1(year, conference):
    assert conference in ["ICLR", "NeurIPS"], "Only ICLR and NeurIPS are supported."

    venue_id = f"{conference}.cc/{year}/Conference"

    submissions = client_v1.get_all_notes(invitation=f"{venue_id}/-/Blind_Submission", details='replies')

    assert len(submissions) > 0, "No submissions found. Something is wrong..."

    dataDir = "openReviewData/raw"

    with open(f'{dataDir}/{conference}{year}.pkl', 'wb') as f:
        pickle.dump(submissions, f)


In [28]:
crawlAPIv1(2021, "NeurIPS")
crawlAPIv1(2022, "NeurIPS")
crawlAPIv2(2023, "NeurIPS")
crawlAPIv2(2024, "NeurIPS")
crawlAPIv2(2025, "NeurIPS")

crawlAPIv1(2018, "ICLR")
crawlAPIv1(2019, "ICLR")
crawlAPIv1(2020, "ICLR")
crawlAPIv1(2021, "ICLR")
crawlAPIv1(2022, "ICLR")
crawlAPIv1(2023, "ICLR")
crawlAPIv2(2024, "ICLR")
crawlAPIv2(2025, "ICLR")

## Get JSON format for each (venue, year)

In [44]:
def saveJSONFormat(year, conference):
    dataDir = "openReviewData/raw"
    submissions = loadSubmissions(year, conference)

    submissionsInJSON = []
    for submission in submissions:
        submissionInJSON = submission.to_json()
        submissionInJSON['details'] = submission.details
        submissionsInJSON.append(submissionInJSON)
    
    with open(f'{dataDir}/{conference}{year}.json', 'w') as f:
        json.dump(submissionsInJSON, f, indent=4)

In [46]:
saveJSONFormat(2018, "ICLR")
saveJSONFormat(2019, "ICLR")
saveJSONFormat(2020, "ICLR")
saveJSONFormat(2021, "ICLR")
saveJSONFormat(2022, "ICLR")
saveJSONFormat(2023, "ICLR")
saveJSONFormat(2024, "ICLR")
saveJSONFormat(2025, "ICLR")
saveJSONFormat(2021, "NeurIPS")
saveJSONFormat(2022, "NeurIPS")
saveJSONFormat(2023, "NeurIPS")
saveJSONFormat(2024, "NeurIPS")
saveJSONFormat(2025, "NeurIPS")

# Filtering and saving raw data

In [14]:
def loadSubmissions(year, conference):
    dataDir = "openReviewData/raw"
    with open(f'{dataDir}/{conference}{year}.pkl', 'rb') as f:
        submissions = pickle.load(f)
    return submissions

In [15]:
submissionsICLR2018 = loadSubmissions(2018, "ICLR")
submissionsICLR2019 = loadSubmissions(2019, "ICLR")
submissionsICLR2020 = loadSubmissions(2020, "ICLR")
submissionsICLR2021 = loadSubmissions(2021, "ICLR")
submissionsICLR2022 = loadSubmissions(2022, "ICLR")
submissionsICLR2023 = loadSubmissions(2023, "ICLR")
submissionsICLR2024 = loadSubmissions(2024, "ICLR")
submissionsICLR2025 = loadSubmissions(2025, "ICLR")

In [16]:
submissionsNeurIPS2021 = loadSubmissions(2021, "NeurIPS")
submissionsNeurIPS2022 = loadSubmissions(2022, "NeurIPS")
submissionsNeurIPS2023 = loadSubmissions(2023, "NeurIPS")
submissionsNeurIPS2024 = loadSubmissions(2024, "NeurIPS")
submissionsNeurIPS2025 = loadSubmissions(2025, "NeurIPS")

# Access replies using submissionsNeurIPS2022[0].details['replies']

In [17]:
def extractReviewSpecificFields(replyContent):
    normalizedFormatContent = {}
    for field, value in replyContent.items():
        normalizedFormatContent[field.replace("_", " ").title()] = value if not isinstance(value, dict) else value['value']
    return normalizedFormatContent


def extractData(submissions, year):
    openReviewURL = 'https://openreview.net'
    output = defaultdict(dict)
    weird_submissions = []

    for submission in submissions:
        if year >= 2024:
            # APIv2 has a different structure
            # Skip all papers which are not accepted or rejected.
            venueid = submission.content['venueid'] if not isinstance(submission.content['venueid'], dict) else submission.content['venueid']['value']
            if "Desk_Rejected" in venueid or "Withdrawn" in venueid:
                continue

        output[submission.id]['Year'] = year
        output[submission.id]['Id'] = submission.id
        output[submission.id]['Title'] = submission.content['title'] if not isinstance(submission.content['title'], dict) else submission.content['title']['value']
        output[submission.id]['Abstract'] = submission.content['abstract'] if not isinstance(submission.content['abstract'], dict) else submission.content['abstract']['value']
        if'pdf' in submission.content:
            output[submission.id]['PDF_URL'] = openReviewURL + submission.content['pdf'] if not isinstance(submission.content['pdf'], dict) else openReviewURL + submission.content['pdf']['value']
        else:
            output[submission.id]['PDF_URL'] = None
        output[submission.id]['Reviews'] = defaultdict(dict)
        for reply in submission.details['replies']:
            if 'metareview' in reply['content']:
                if year == 2019:
                    # Only for ICLR 2019
                    output[submission.id]['Decision'] = reply['content']['recommendation']
                elif 'decision' in reply['content']:
                    output[submission.id]['Decision'] = reply['content']['decision'] if not isinstance(reply['content']['decision'], dict) else reply['content']['decision']['value']
                continue  # Skip metareviews except for decision extraction

            if 'rating' in reply['content'] or 'recommendation' in reply['content']:
                output[submission.id]['Reviews'][reply['id']] = extractReviewSpecificFields(reply['content'])
                output[submission.id]['Reviews'][reply['id']]['writer'] = reply['signatures'][0].split('/')[-1]
            
            if 'decision' in reply['content']:
                output[submission.id]['Decision'] = reply['content']['decision'] if not isinstance(reply['content']['decision'], dict) else reply['content']['decision']['value']

            if 'withdrawal_confirmation' in reply['content']:
                output[submission.id]['Decision'] = 'Withdrawn'

            if 'desk_reject_comments' in reply['content']:
                output[submission.id]['Decision'] = 'Desk Rejected'

        if "Decision" not in output[submission.id]:
            output[submission.id]['Decision'] = None
            weird_submissions.append(submission.id)
        # assert "Decision" in output[submission.id], f"Decision missing for {submission.id}"
    print(f"Total Submissions without decision: {len(weird_submissions)}")
    return output

In [22]:
# Save dictionary for each year as JSON file
outputDir = "openReviewData/processed"

with open(f'{outputDir}/NeurIPS-2021.json', 'w') as f:
    json.dump(extractData(submissionsNeurIPS2021, 2021), f, indent=4)
with open(f'{outputDir}/NeurIPS-2022.json', 'w') as f:
    json.dump(extractData(submissionsNeurIPS2022, 2022), f, indent=4)
with open(f'{outputDir}/NeurIPS-2023.json', 'w') as f:
    json.dump(extractData(submissionsNeurIPS2023, 2023), f, indent=4)
with open(f'{outputDir}/NeurIPS-2024.json', 'w') as f:
    json.dump(extractData(submissionsNeurIPS2024, 2024), f, indent=4)
with open(f'{outputDir}/NeurIPS-2025.json', 'w') as f:
    json.dump(extractData(submissionsNeurIPS2025, 2025), f, indent=4)

Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0


In [20]:
# Save dictionary for each year as JSON file
outputDir = "openReviewData/processed"

with open(f'{outputDir}/ICLR-2018.json', 'w') as f:
    json.dump(extractData(submissionsICLR2018, 2018), f, indent=4)

with open(f'{outputDir}/ICLR-2019.json', 'w') as f:
    json.dump(extractData(submissionsICLR2019, 2019), f, indent=4)

with open(f'{outputDir}/ICLR-2020.json', 'w') as f:
    json.dump(extractData(submissionsICLR2020, 2020), f, indent=4)

with open(f'{outputDir}/ICLR-2021.json', 'w') as f:
    json.dump(extractData(submissionsICLR2021, 2021), f, indent=4)

with open(f'{outputDir}/ICLR-2022.json', 'w') as f:
    json.dump(extractData(submissionsICLR2022, 2022), f, indent=4)

with open(f'{outputDir}/ICLR-2023.json', 'w') as f:
    json.dump(extractData(submissionsICLR2023, 2023), f, indent=4)

with open(f'{outputDir}/ICLR-2024.json', 'w') as f:
    json.dump(extractData(submissionsICLR2024, 2024), f, indent=4)

with open(f'{outputDir}/ICLR-2025.json', 'w') as f:
    json.dump(extractData(submissionsICLR2025, 2025), f, indent=4)

Total Submissions without decision: 12
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0
Total Submissions without decision: 0


# Preprocessing raw data and saving

In [ ]:
from reviewDataset import NLPeerReviewDataset
from nlpeer import DATASETS

# Assign directory as needed
dataDir = ""

arr22ReviewDataset = NLPeerReviewDataset(DATASETS("ARR-22"), dataDir)
arrEMNLP24ReviewDataset = NLPeerReviewDataset(DATASETS("ARR-EMNLP-2024"), dataDir)
arrNAACL25ReviewDataset = NLPeerReviewDataset(DATASETS("ARR-NAACL-2025"), dataDir)
arrACL25ReviewDataset = NLPeerReviewDataset(DATASETS("ARR-ACL-2025"), dataDir)

In [ ]:
from model.datasets import OPENREVIEW_DATASETS
from reviewDataset import OpenreviewDataset

dataDir = ""

iclrReviewDataset2018 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_18, dataDir)
iclrReviewDataset2019 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_19, dataDir)
iclrReviewDataset2020 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_20, dataDir)
iclrReviewDataset2021 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_21, dataDir)
iclrReviewDataset2022 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_22, dataDir)
iclrReviewDataset2023 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_23, dataDir)
iclrReviewDataset2024 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_24, dataDir)
iclrReviewDataset2025 = OpenreviewDataset(OPENREVIEW_DATASETS.ICLR_25, dataDir)
neurIPSReviewDataset2021 = OpenreviewDataset(OPENREVIEW_DATASETS.NEURIPS_21, dataDir)
neurIPSReviewDataset2022 = OpenreviewDataset(OPENREVIEW_DATASETS.NEURIPS_22, dataDir)
neurIPSReviewDataset2023 = OpenreviewDataset(OPENREVIEW_DATASETS.NEURIPS_23, dataDir)
neurIPSReviewDataset2024 = OpenreviewDataset(OPENREVIEW_DATASETS.NEURIPS_24, dataDir)
neurIPSReviewDataset2025 = OpenreviewDataset(OPENREVIEW_DATASETS.NEURIPS_25, dataDir)